# Violencia contra la mujer - Data Mining

Unidad de análisis: un distrito del País

Variable de interés: Tasa de denuncias por cada 1000 habitantes en el distrito.

In [1]:
# importar librerias

import pandas as pd

## Dataset de población estimada en el país

In [2]:
# cargar datos a dataframe poblacion

poblacion = pd.read_excel('../data/raw/6894980-peru-poblacion-total-proyectada-al-30-de-junio-de-cada-ano-segun-departamento-provincia-y-distrito-2018-2026.xlsx', skiprows = 4)

poblacion.columns = ['UBIGEO', 'Nombre', '2018', '2019', '2020', '2021', '2022', '2023', '2024', '2025', '2026'] # renombramos las columnas


In [3]:
# forzamos conversion a tipo de dato numérico, para obtener nan en pie de página
poblacion['UBIGEO_NUM'] = pd.to_numeric(poblacion['UBIGEO'], errors='coerce')
poblacion = poblacion.dropna(subset=['UBIGEO_NUM']).copy() # eliminar valores nan

# creamos columnas adicionales
poblacion['Departamento'] = None
poblacion['Provincia'] = None

# asignar el nombre de departamento para ubigeos divisibles entre 10000
poblacion.loc[poblacion['UBIGEO_NUM'] % 10000 == 0, 'Departamento'] = poblacion['Nombre']

# asignar el nombre de provincia para ubigeos divisibles  entre 100 pero no entre 10000
es_provincia = (poblacion['UBIGEO_NUM'] % 100 == 0) & (poblacion['UBIGEO_NUM'] % 10000 != 0)
poblacion.loc[es_provincia, 'Provincia'] = poblacion['Nombre']

# rellenar el nombre de departamento y provincia hacia abajo
poblacion['Departamento'] = poblacion['Departamento'].ffill()
poblacion['Provincia'] = poblacion['Provincia'].ffill()

## filtrar por distrito, no divisibles entre 100 y renombrar columna a Distrito
poblacion_distritos = poblacion[poblacion['UBIGEO_NUM'] % 100 != 0].copy()
poblacion_distritos = poblacion_distritos.rename(columns={'Nombre': 'Distrito'})

# transformar el los valores de "ancho" a "largo"
columnas_identificadoras = ['Departamento', 'Provincia', 'Distrito', 'UBIGEO']
años = ['2018', '2019', '2020', '2021', '2022', '2023', '2024', '2025', '2026']

poblacion_final = pd.melt(
    poblacion_distritos,
    id_vars=columnas_identificadoras,
    value_vars=años,
    var_name='Año',
    value_name='Población'
)

# volver al ubigeo de 6 cifras, puede empezar en 0
poblacion_final['UBIGEO'] = pd.to_numeric(poblacion_final['UBIGEO']).astype(int).astype(str).str.zfill(6)

# convertir la columna Población a tipo de dato numérico
poblacion_final['Población'] = pd.to_numeric(poblacion_final['Población'], errors='coerce')

In [4]:
poblacion_final.info()

<class 'pandas.DataFrame'>
RangeIndex: 17028 entries, 0 to 17027
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Departamento  17028 non-null  str    
 1   Provincia     17028 non-null  str    
 2   Distrito      17028 non-null  str    
 3   UBIGEO        17028 non-null  str    
 4   Año           17028 non-null  str    
 5   Población     16969 non-null  float64
dtypes: float64(1), str(5)
memory usage: 798.3 KB


Existen valores nulos en `Población` debido a la existencia de distritos creados posteriormente al año base (2018).

In [5]:
poblacion_final.describe().round()

,Población
count,16969.0
mean,17651.0
std,58188.0
min,104.0
25%,1487.0
50%,3863.0
75%,10722.0
max,1300785.0


In [6]:
# nombres de las variables a minúsculas
poblacion_final.columns = poblacion_final.columns.str.lower()

In [7]:
print(f'El dataframe tiene {poblacion_final.shape[0]} filas y {poblacion_final.shape[1]} columnas, ')
print(f'Los valores faltantes por columna es:\n{poblacion_final.isna().sum()}')
print(f'El registro del distrito con mayor población es:\n{poblacion_final[poblacion_final.población == poblacion_final.población.max()]}')

El dataframe tiene 17028 filas y 6 columnas, 
Los valores faltantes por columna es:
departamento     0
provincia        0
distrito         0
ubigeo           0
año              0
población       59
dtype: int64
El registro del distrito con mayor población es:
      departamento           provincia                distrito  ubigeo   año  \
16460        LIMA   LIMA METROPOLITANA  SAN JUAN DE LURIGANCHO  150132  2026   

       población  
16460  1300785.0  


In [8]:
poblacion_final['ubigeo'] = poblacion_final['ubigeo'].astype(str).str.zfill(6)
poblacion_final.rename(columns={'año': 'anio'}, inplace=True)

In [10]:
poblacion_final.to_csv('../data/processed/poblacion_por_distrito.csv', index=False)

In [8]:
poblacion_final

,Departamento,Provincia,Distrito,UBIGEO,Año,Población
0,AMAZONAS,CHACHAPOYAS,CHACHAPOYAS,010101,2018,36602.0
1,AMAZONAS,CHACHAPOYAS,ASUNCION,010102,2018,283.0
2,AMAZONAS,CHACHAPOYAS,BALSAS,010103,2018,1222.0
3,AMAZONAS,CHACHAPOYAS,CHETO,010104,2018,687.0
4,AMAZONAS,CHACHAPOYAS,CHILIQUIN,010105,2018,619.0
...,...,...,...,...,...,...
17023,UCAYALI,PADRE ABAD,NESHUYA,250304,2026,12661.0
17024,UCAYALI,PADRE ABAD,ALEXANDER VON HUMBOLDT,250305,2026,6811.0
17025,UCAYALI,PADRE ABAD,HUIPOCA 6/,250306,2026,5010.0
17026,UCAYALI,PADRE ABAD,BOQUERON 11/,250307,2026,5887.0
